# 07 - CI/CD Pipeline (SageMaker SDK v3)

Builds the complete v3 DAG directly in this notebook:

`Preprocess -> Train -> Evaluate -> QualityGate -> Register / Fail`

It no longer imports the repository's v2 `pipelines/pipeline.py`. The training and inference framework version is aligned with notebook 04.


In [2]:
import sys, json, os, shutil, tempfile
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import boto3
import pandas as pd

from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.processing import FrameworkProcessor
from sagemaker.core.shapes import ProcessingInput, ProcessingS3Input, ProcessingOutput, ProcessingS3Output
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core.workflow.parameters import ParameterFloat, ParameterString
from sagemaker.core.model_metrics import MetricsSource, ModelMetrics
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, InputData, OutputDataConfig, SourceCode
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import ProcessingStep, TrainingStep, CacheConfig
from sagemaker.mlops.workflow.model_step import ModelStep
from sagemaker.mlops.workflow.condition_step import ConditionStep
#from sagemaker.mlops.workflow.conditions import ConditionGreaterThanOrEqualTo, ConditionLessThanOrEqualTo
from sagemaker.mlops.workflow.fail_step import FailStep
#from sagemaker.mlops.workflow.functions import Join, JsonGet
#from sagemaker.mlops.workflow.properties import PropertyFile

from sagemaker.core.workflow.conditions import (
    ConditionGreaterThanOrEqualTo,
    ConditionLessThanOrEqualTo,
)
from sagemaker.core.workflow.functions import Join, JsonGet
from sagemaker.core.workflow.properties import PropertyFile

from src import config

session = Session()
pipeline_session = PipelineSession(default_bucket=os.environ.get("CHURN_BUCKET") or session.default_bucket())
role = get_execution_role()
REGION = session.boto_region_name
BUCKET = os.environ.get("CHURN_BUCKET", session.default_bucket())
FRAMEWORK_VERSION = "1.4-2-py312"
INSTANCE_TYPE = "ml.m5.xlarge"

def stage_source():
    build = Path(tempfile.mkdtemp(prefix="churn-pipeline-v3-"))
    shutil.copytree(REPO_ROOT / "src", build / "src", ignore=shutil.ignore_patterns("__pycache__"))
    for script in (REPO_ROOT / "scripts").glob("*.py"):
        shutil.copy2(script, build / script.name)
    (build / "requirements.txt").write_text("pyarrow\npandas\nscikit-learn\njoblib\n")
    return str(build)

def load_best_params():
    key = f"{config.MODEL_PREFIX}/best_params.json"
    try:
        body = boto3.client("s3", region_name=REGION).get_object(Bucket=BUCKET, Key=key)["Body"].read()
        return json.loads(body)
    except Exception as error:
        if getattr(error, "response", {}).get("Error", {}).get("Code") not in ("NoSuchKey", "404"):
            raise
        return {"model_type": "gb", "hyperparameters": {}}

best = load_best_params()
source = stage_source()
image_uri = image_uris.retrieve(
    framework="sklearn", region=REGION, version=FRAMEWORK_VERSION,
    py_version="py3", instance_type=INSTANCE_TYPE,
)
print("v3 pipeline environment:", REGION, BUCKET, image_uri)


v3 pipeline environment: us-east-1 sagemaker-us-east-1-025824828736 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-py312-cpu-py3


## 1. Define the v3 pipeline


In [3]:
min_auc = ParameterFloat(name="MinRocAuc", default_value=config.MIN_ROC_AUC)
min_recall = ParameterFloat(name="MinRecall", default_value=config.MIN_RECALL)
max_gap = ParameterFloat(name="MaxFairnessGap", default_value=config.MAX_FAIRNESS_GAP)
model_type = ParameterString(name="ModelType", default_value=best.get("model_type", "gb"))
cache = CacheConfig(enable_caching=True, expire_after="30d")

raw = f"s3://{BUCKET}/{config.RAW_PREFIX}"
processed = f"s3://{BUCKET}/{config.PROCESSED_PREFIX}/pipeline"

processor = FrameworkProcessor(
    image_uri=image_uri, command=["python3"], role=role,
    instance_type=INSTANCE_TYPE, instance_count=1,
    base_job_name="churn-preprocess-v3", sagemaker_session=pipeline_session,
)
prep_args = processor.run(
    code="preprocess.py", source_dir=source, requirements="requirements.txt",
    inputs=[
        ProcessingInput(input_name="raw-train", s3_input=ProcessingS3Input(
            s3_uri=f"{raw}/train/", local_path="/opt/ml/processing/input/raw/train",
            s3_data_type="S3Prefix", s3_input_mode="File")),
        ProcessingInput(input_name="raw-test", s3_input=ProcessingS3Input(
            s3_uri=f"{raw}/test/", local_path="/opt/ml/processing/input/raw/test",
            s3_data_type="S3Prefix", s3_input_mode="File")),
    ],
    outputs=[ProcessingOutput(output_name=name, s3_output=ProcessingS3Output(
        s3_uri=f"{processed}/{name}", local_path=f"/opt/ml/processing/output/{name}",
        s3_upload_mode="EndOfJob")) for name in ("train", "test", "production")],
)
step_prep = ProcessingStep(name="Preprocess", step_args=prep_args, cache_config=cache)

hyperparameters = {**best.get("hyperparameters", {}), "model-type": model_type}
trainer = ModelTrainer(
    training_image=image_uri,
    source_code=SourceCode(source_dir=source, entry_script="train.py", requirements="requirements.txt"),
    compute=Compute(instance_type=INSTANCE_TYPE, instance_count=1),
    role=role,
    base_job_name="churn-train-v3",
    sagemaker_session=pipeline_session,
    output_data_config=OutputDataConfig(s3_output_path=f"s3://{BUCKET}/{config.MODEL_PREFIX}/pipeline"),
    hyperparameters=hyperparameters,
    input_data_config=[
        InputData(channel_name="train", data_source=step_prep.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
                  content_type="application/x-parquet"),
        InputData(channel_name="test", data_source=step_prep.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
                  content_type="application/x-parquet"),
    ],
)
step_train = TrainingStep(name="Train", step_args=trainer.train(), cache_config=cache)

report = PropertyFile(name="EvaluationReport", output_name="evaluation", path="evaluation.json")
eval_processor = FrameworkProcessor(
    image_uri=image_uri, command=["python3"], role=role,
    instance_type=INSTANCE_TYPE, instance_count=1,
    base_job_name="churn-evaluate-v3", sagemaker_session=pipeline_session,
)
eval_args = eval_processor.run(
    code="evaluate.py", source_dir=source, requirements="requirements.txt",
    inputs=[
        ProcessingInput(input_name="model", s3_input=ProcessingS3Input(
            s3_uri=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            local_path="/opt/ml/processing/model", s3_data_type="S3Prefix", s3_input_mode="File")),
        ProcessingInput(input_name="test", s3_input=ProcessingS3Input(
            s3_uri=step_prep.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            local_path="/opt/ml/processing/test", s3_data_type="S3Prefix", s3_input_mode="File")),
    ],
    outputs=[ProcessingOutput(output_name="evaluation", s3_output=ProcessingS3Output(
        s3_uri=f"s3://{BUCKET}/{config.MODEL_PREFIX}/pipeline/evaluation",
        local_path="/opt/ml/processing/evaluation", s3_upload_mode="EndOfJob"))],
)
step_eval = ProcessingStep(name="Evaluate", step_args=eval_args, property_files=[report], cache_config=cache)

metrics_uri = Join(on="/", values=[
    step_eval.properties.ProcessingOutputConfig.Outputs["evaluation"].S3Output.S3Uri,
    "evaluation.json",
])
model_builder = ModelBuilder(
    s3_model_data_url=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    image_uri=image_uri,
    source_code=SourceCode(source_dir=source, entry_script="inference.py", requirements="requirements.txt"),
    role_arn=role, sagemaker_session=pipeline_session,
)
step_register = ModelStep(
    name="RegisterChurnModel",
    step_args=model_builder.register(
        model_package_group_name=config.MODEL_PACKAGE_GROUP,
        content_types=["text/csv"], response_types=["text/csv"],
        inference_instances=["ml.m5.large"], transform_instances=["ml.m5.large"],
        approval_status="PendingManualApproval",
        model_metrics=ModelMetrics(model_statistics=MetricsSource(
            s3_uri=metrics_uri, content_type="application/json")),
    ),
)

def metric(path):
    return JsonGet(step_name=step_eval.name, property_file=report, json_path=path)

step_fail = FailStep(
    name="QualityGateFailed",
    error_message="Model did not meet ROC-AUC, recall, or fairness thresholds",
)
step_gate = ConditionStep(
    name="QualityGate",
    conditions=[
        ConditionGreaterThanOrEqualTo(left=metric("metrics.roc_auc.value"), right=min_auc),
        ConditionGreaterThanOrEqualTo(left=metric("metrics.recall.value"), right=min_recall),
        ConditionLessThanOrEqualTo(left=metric("fairness_max_gap.gender"), right=max_gap),
        ConditionLessThanOrEqualTo(left=metric("fairness_max_gap.age_band"), right=max_gap),
    ],
    if_steps=[step_register], else_steps=[step_fail],
)

pipeline = Pipeline(
    name=config.PIPELINE_NAME,
    parameters=[min_auc, min_recall, max_gap, model_type],
    steps=[step_prep, step_train, step_eval, step_gate],
    sagemaker_session=pipeline_session,
)
pipeline.upsert(role_arn=role)
print("pipeline:", pipeline.name)


[10/19/26 12:59:03] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=2212493;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=2212494;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#322\322]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


                    INFO     StoppingCondition not provided. Using default:                         ]8;id=2212501;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=2212502;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=2212508;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=2212509;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#165\165]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=2212516;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=2212517;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-lea                     
                             rn:1.4-2-py312-cpu-py3                                                                

...

                    WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=2212553;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=2212554;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=2212559;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=2212560;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'CertifyForMarketplace' from the pipeline definition     ]8;id=2212566;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py\model_step.py]8;;\:]8;id=2212567;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py#195\195]8;;\
                             since it will be overridden in pipeline execution time.                               

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=2212572;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=2212573;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

pipeline: churn-training-pipeline


## 2. Inspect the DAG


In [4]:
definition = json.loads(pipeline.definition())
for step in definition["Steps"]:
    print(f"{step['Name']:22s} {step['Type']:14s} depends on: {step.get('DependsOn', '-')}")


[10/19/26 12:59:27] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=2212578;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=2212579;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[10/19/26 12:59:28] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=2212584;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=2212585;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=2212590;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=2212591;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=2212596;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=2212597;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

Preprocess             Processing     depends on: -
Train                  Training       depends on: -
Evaluate               Processing     depends on: -
QualityGate            Condition      depends on: -


## 3. Successful run


In [ ]:
success = pipeline.start(parameters={"ModelType": best.get("model_type", "gb")})
print(success.arn)
try:
    success.wait(delay=30, max_attempts=120)
except Exception as error:
    print("run did not succeed:", error)

def show_steps(execution):
    return pd.DataFrame([
        {"step": item["StepName"], "status": item["StepStatus"], "reason": item.get("FailureReason", "")}
        for item in execution.list_steps()
    ])

show_steps(success)


arn:aws:sagemaker:us-east-1:025824828736:pipeline/churn-training-pipeline/execution/2xz6rt7a7ulo


## 4. Intentional quality-gate failure


In [ ]:
failed = pipeline.start(parameters={"MinRocAuc": 0.999, "ModelType": best.get("model_type", "gb")})
try:
    failed.wait(delay=30, max_attempts=120)
except Exception as error:
    print("expected failure:", str(error)[:240])
show_steps(failed)


## 5. Run an alternative model and manually approve a package


In [ ]:
improved = pipeline.start(parameters={"ModelType": "rf"})
print("alternative run:", improved.arn)


In [ ]:
sm = boto3.client("sagemaker", region_name=REGION)
pending = sm.list_model_packages(
    ModelPackageGroupName=config.MODEL_PACKAGE_GROUP,
    ModelApprovalStatus="PendingManualApproval",
    SortBy="CreationTime", SortOrder="Descending",
)["ModelPackageSummaryList"]
display(pd.DataFrame(pending)[["ModelPackageVersion", "ModelApprovalStatus", "CreationTime"]] if pending else "nothing pending")

# Set APPROVE_LATEST=True only after reviewing evaluation metrics.
APPROVE_LATEST = False
if APPROVE_LATEST and pending:
    sm.update_model_package(
        ModelPackageArn=pending[0]["ModelPackageArn"],
        ModelApprovalStatus="Approved",
    )
    print("approved version", pending[0]["ModelPackageVersion"])


## 6. Local CI checks


In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=REPO_ROOT, check=True)


## IAM requirements

The notebook role needs permission to create/update SageMaker pipelines, start executions, create processing and training jobs, register model packages, pass the SageMaker execution role, and read/write the configured S3 prefixes. GitHub OIDC deployment also needs `AWS_ROLE_ARN`, `SAGEMAKER_ROLE_ARN`, `CHURN_BUCKET`, and `AWS_REGION`.


In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>


In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>